# BD-KDD Kidney Disease — LASSO + DODA Feature Selection

## Experimental objective

Evaluate conventional **LASSO feature selection** against **LASSO + DODA clinical-priority re-ranking** on the BD-KDD kidney-disease dataset.

### Important design rule

LASSO scores **all 24 predictors**. DODA also receives **all 24 predictor scores** and performs the final Top-K selection after clinical-priority fusion.

This is essential: if LASSO selected Top-K before DODA, DODA could not re-prioritize features outside that preliminary Top-K.

### Dataset

- 988 records
- 24 predictors
- Binary target: `Class`
- `Class = 0`: healthy
- `Class = 1`: kidney disease
- 0 missing values
- 80/20 stratified fixed split for the initial experiment
- 5×5 repeated stratified CV for robustness


In [3]:
# =============================================================================
# STEP 1: LOAD PROCESSED DATASET
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/processed/BD-KDD_cleaned.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("PROCESSED BD-KDD DATASET")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


PROCESSED BD-KDD DATASET
Rows    : 988
Columns : 25


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane,Class
0,58,147,1.025,1,3,0,1,0,0,177,...,43,12342,4.3,1,0,0,1,1,0,0
1,71,142,1.025,2,3,1,0,1,0,247,...,47,7249,3.7,0,1,1,0,1,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,37,4111,3.7,0,0,0,0,0,1,0
3,34,70,1.005,1,3,1,1,0,0,215,...,44,4682,4.0,1,0,1,1,0,1,1
4,62,120,1.020,4,0,1,0,1,0,143,...,54,5161,4.7,0,0,1,0,1,0,1


In [4]:
# =============================================================================
# STEP 2: FEATURE AND TARGET SEPARATION
# =============================================================================

TARGET_COLUMN = "Class"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)
print(f"Predictors (X) : {X.shape[1]}")
print(f"Records        : {X.shape[0]}")
print(f"Target (y)     : {TARGET_COLUMN}")
print(f"Target values  : {sorted(y.unique().tolist())}")

assert X.shape[1] == 24, "Expected 24 predictors after removing Class."
assert set(y.unique()) == {0, 1}

display(X.head())
display(y.head())


FEATURE / TARGET SEPARATION
Predictors (X) : 24
Records        : 988
Target (y)     : Class
Target values  : [0, 1]


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Hemo,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane
0,58,147,1.025,1,3,0,1,0,0,177,...,9.3,43,12342,4.3,1,0,0,1,1,0
1,71,142,1.025,2,3,1,0,1,0,247,...,16.9,47,7249,3.7,0,1,1,0,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,16.6,37,4111,3.7,0,0,0,0,0,1
3,34,70,1.005,1,3,1,1,0,0,215,...,15.4,44,4682,4.0,1,0,1,1,0,1
4,62,120,1.020,4,0,1,0,1,0,143,...,13.9,54,5161,4.7,0,0,1,0,1,0


0    0
1    1
2    0
3    1
4    1
Name: Class, dtype: int64

In [5]:
# =============================================================================
# STEP 3: TARGET DISTRIBUTION
# =============================================================================

target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(target_distribution)


,Count,Percentage
Class,,
0,481,48.68
1,507,51.32


In [6]:
# =============================================================================
# STEP 4: DATA TYPE AND MISSING-VALUE CHECK
# =============================================================================

print("=" * 70)
print("DATA TYPES")
print("=" * 70)
display(X.dtypes.to_frame("Data Type"))

missing = pd.DataFrame({
    "Missing Count": X.isna().sum(),
    "Missing Percentage (%)": (X.isna().mean() * 100).round(2)
})

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing)

assert X.isna().sum().sum() == 0, "Unexpected missing values detected."

print("No missing values detected. No imputation will be performed.")


DATA TYPES


,Data Type
Age,int64
Bp,int64
Sg,float64
Al,int64
Su,int64
Rbc,int64
Pc,int64
Pcc,int64
Ba,int64
Bgr,int64


MISSING VALUES


,Missing Count,Missing Percentage (%)
Age,0,0.0
Bp,0,0.0
Sg,0,0.0
Al,0,0.0
Su,0,0.0
Rbc,0,0.0
Pc,0,0.0
Pcc,0,0.0
Ba,0,0.0
Bgr,0,0.0


No missing values detected. No imputation will be performed.


## Preprocessing decision

The EDA established that BD-KDD contains **zero missing values**. Therefore, no imputation step is included in this notebook.

Feature selection is fitted only on training data/folds. Model scaling is applied only where needed and is fitted only on training data/folds.


In [7]:
# =============================================================================
# STEP 5: FIXED 80/20 STRATIFIED TRAIN-TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nTraining class distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting class distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))


TRAIN-TEST SPLIT
X_train : (790, 24)
X_test  : (198, 24)
y_train : (790,)
y_test  : (198,)

Training class distribution:


Class
1    51.27
0    48.73
Name: proportion, dtype: float64


Testing class distribution:


Class
1    51.52
0    48.48
Name: proportion, dtype: float64

In [8]:
# =============================================================================
# STEP 6: FEATURE-SCALE CHECK
# =============================================================================
# LASSO F-statistics are scale-invariant, so feature selection is performed
# on the original training-fold values.
#
# Scaling is applied later only for Logistic Regression.

print("LASSO will be fitted on the original training data.")
print("Logistic Regression will be scaled after feature selection.")
print("Random Forest and XGBoost will use the selected features without scaling.")


LASSO will be fitted on the original training data.
Logistic Regression will be scaled after feature selection.
Random Forest and XGBoost will use the selected features without scaling.


# 1. LASSO Baseline

LASSO is used as the statistical feature-selection baseline.

**Important:** LASSO is fitted on all 24 predictors first. Feature importance is based on the absolute L1 coefficients, and Top-K is applied only after the complete ranking has been generated.


In [9]:
# =============================================================================
# STEP 7: LASSO SCORES FOR ALL FEATURES
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

lasso_scaler = StandardScaler()

X_train_lasso_scaled = pd.DataFrame(
    lasso_scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_lasso_scaled = pd.DataFrame(
    lasso_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

lasso_selector = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

lasso_selector.fit(
    X_train_lasso_scaled,
    y_train
)

lasso_scores = pd.DataFrame({
    "Feature": X_train.columns,
    "LASSO_Coefficient": lasso_selector.coef_[0],
    "LASSO_Absolute_Coefficient": np.abs(lasso_selector.coef_[0])
}).sort_values(
    "LASSO_Absolute_Coefficient",
    ascending=False
).reset_index(drop=True)

lasso_scores["LASSO_Rank"] = np.arange(
    1,
    len(lasso_scores) + 1
)

display(lasso_scores)


c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,Feature,LASSO_Coefficient,LASSO_Absolute_Coefficient,LASSO_Rank
0,Bp,-0.162750,0.162750,1
1,Bu,0.038219,0.038219,2
2,Su,-0.033818,0.033818,3
3,Ba,-0.033182,0.033182,4
4,Ane,0.016304,0.016304,5
5,Appet,-0.015208,0.015208,6
6,Pcv,-0.007970,0.007970,7
7,Bgr,0.006839,0.006839,8
8,Sc,0.005209,0.005209,9
9,Dm,0.002901,0.002901,10


In [10]:
# =============================================================================
# STEP 8: TOP-K LASSO FEATURE SETS
# =============================================================================

TOP_K_VALUES = [5, 10, 15, 20]

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")

lasso_results = {}

for top_k in TOP_K_VALUES:

    selected = (
        lasso_scores
        .head(top_k)["Feature"]
        .tolist()
    )

    lasso_results[top_k] = {
        "features": selected,
        "X_train": X_train[selected].copy(),
        "X_test": X_test[selected].copy()
    }

    print("=" * 70)
    print(f"LASSO TOP-{top_k}")
    print("=" * 70)
    print(selected)


Total predictors: 24
Top-K values: [5, 10, 15, 20]
LASSO TOP-5
['Bp', 'Bu', 'Su', 'Ba', 'Ane']
LASSO TOP-10
['Bp', 'Bu', 'Su', 'Ba', 'Ane', 'Appet', 'Pcv', 'Bgr', 'Sc', 'Dm']
LASSO TOP-15
['Bp', 'Bu', 'Su', 'Ba', 'Ane', 'Appet', 'Pcv', 'Bgr', 'Sc', 'Dm', 'Sg', 'Rbcc', 'Age', 'Pc', 'Rbc']
LASSO TOP-20
['Bp', 'Bu', 'Su', 'Ba', 'Ane', 'Appet', 'Pcv', 'Bgr', 'Sc', 'Dm', 'Sg', 'Rbcc', 'Age', 'Pc', 'Rbc', 'Al', 'Pcc', 'Hemo', 'Sod', 'Pot']


In [11]:
# =============================================================================
# STEP 9: DOWNSTREAM MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print(list(models.keys()))


['Logistic Regression', 'Random Forest', 'XGBoost']


In [12]:
# =============================================================================
# STEP 10: LASSO BASELINE MODEL EVALUATION
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

fixed_lasso_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = lasso_results[top_k]["X_train"]
    Xte_selected = lasso_results[top_k]["X_test"]
    selected_features = lasso_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_lasso_rows.append({
            "Method": "LASSO",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_lasso_results_df = pd.DataFrame(fixed_lasso_rows)
display(fixed_lasso_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,LASSO,5,Logistic Regression,"Bp, Bu, Su, Ba, Ane",0.5303,0.5391,0.6078,0.5714,0.5587
1,LASSO,5,Random Forest,"Bp, Bu, Su, Ba, Ane",0.5354,0.5532,0.5098,0.5306,0.5628
2,LASSO,5,XGBoost,"Bp, Bu, Su, Ba, Ane",0.5455,0.5577,0.5686,0.5631,0.5288
3,LASSO,10,Logistic Regression,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm",0.5455,0.5517,0.6275,0.5872,0.5642
4,LASSO,10,Random Forest,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm",0.5505,0.5644,0.5588,0.5616,0.5575
5,LASSO,10,XGBoost,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm",0.5253,0.5385,0.5490,0.5437,0.5239
6,LASSO,15,Logistic Regression,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.5758,0.5818,0.6275,0.6038,0.5774
7,LASSO,15,Random Forest,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.5101,0.5234,0.5490,0.5359,0.5283
8,LASSO,15,XGBoost,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.4545,0.4722,0.5000,0.4857,0.4861
9,LASSO,20,Logistic Regression,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.5657,0.5755,0.5980,0.5865,0.5751


In [13]:
# =============================================================================
# STEP 11: SAVE LASSO BASELINE
# =============================================================================

baseline_dir = Path("../../../results/kidney_disease/baseline")
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_path = baseline_dir / "lasso_baseline_results.csv"
fixed_lasso_results_df.to_csv(baseline_path, index=False)

print(f"Saved: {baseline_path.resolve()}")


Saved: C:\Users\johnm\msc_research\results\kidney_disease\baseline\lasso_baseline_results.csv


# 2. DODA Clinical Re-ranking

### Correct experimental order

```text
24 original predictors
        ↓
LASSO coefficients for all 24
        ↓
statistical ranking
        +
pre-specified clinical weights
        ↓
DODA rank fusion
        ↓
final Top-K
```

DODA must **not** receive an already-truncated LASSO Top-K set. Otherwise, clinically prioritized features outside that preliminary Top-K would be impossible to recover.


In [14]:
# =============================================================================
# STEP 12: INSTALL / IMPORT DODA
# =============================================================================

# Run the following installation once if DODA is not installed:
# %pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

import doda

from doda import DODASelector
from doda.adapters import SklearnAdapter
from doda.knowledge.providers import JSONProvider
from doda.fusion import RankFusion

print("DODA:", doda.__file__)

print("Provider:", JSONProvider)
print("Fusion:", RankFusion)

DODA: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\__init__.py
Provider: <class 'doda.knowledge.providers.json_provider.JSONProvider'>
Fusion: <class 'doda.fusion.rank.RankFusion'>


In [15]:
# =============================================================================
# STEP 13: LOCATE AND VALIDATE CLINICAL WEIGHTS
# =============================================================================

from pathlib import Path
from doda.knowledge.providers import JSONProvider

WEIGHTS_FILENAME = "BD_KDD_clinical_weights.json"

search_roots = [Path.cwd(), *Path.cwd().parents]
matches = []

for root in search_roots:
    try:
        matches.extend(root.rglob(WEIGHTS_FILENAME))
    except (PermissionError, OSError):
        continue

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{WEIGHTS_FILENAME} was not found. "
        "Place it in the repository and rerun this cell."
    )

WEIGHTS_PATH = matches[0]

print(f"Using clinical weights: {WEIGHTS_PATH.resolve()}")

# Use the standard DODA JSON provider
provider = JSONProvider(str(WEIGHTS_PATH))

# Validate coverage using the flat JSON directly
import json

with open(WEIGHTS_PATH, "r", encoding="utf-8") as f:
    clinical_weights = json.load(f)

missing_weights = sorted(
    set(X.columns) - set(clinical_weights)
)

extra_weights = sorted(
    set(clinical_weights) - set(X.columns)
)

print(f"Clinical weights: {len(clinical_weights)}")
print(f"Predictors:       {len(X.columns)}")
print(f"Missing weights:  {missing_weights}")
print(f"Extra weights:    {extra_weights}")

assert not missing_weights
assert not extra_weights

Using clinical weights: C:\Users\johnm\msc_research\config\clinical_weights\BD_KDD_clinical_weights.json
Clinical weights: 24
Predictors:       24
Missing weights:  []
Extra weights:    []


In [16]:
# =============================================================================
# STEP 14: DODA — SCORE ALL FEATURES, THEN APPLY TOP-K
# =============================================================================

rank_doda_results = {}

for top_k in TOP_K_VALUES:

    lasso_operator = SklearnAdapter(
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.1,
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )

    doda_selector = DODASelector(
        operators=[lasso_operator],
        provider=provider,
        fusion=RankFusion(),
        top_k=top_k
    )

    # Fit on the scaled training data only.
    doda_selector.fit(
        X_train_lasso_scaled,
        y_train
    )

    selected_features = list(
        doda_selector.get_selected_features()
    )

    rank_doda_results[top_k] = {
        "selector": doda_selector,
        "features": selected_features,
        "X_train": X_train[selected_features].copy(),
        "X_test": X_test[selected_features].copy()
    }

    print("=" * 70)
    print(f"DODA TOP-{top_k}")
    print("=" * 70)
    print(selected_features)
    print(f"Number selected: {len(selected_features)}")

    assert len(selected_features) == top_k


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7F98AC980>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.0), 'Bp': np.float64(0.16275005422941846), 'Sg': np.float64(0.0025025929452409832), 'Al': np.float64(0.0), 'Su': np.float64(0.03381782645035158), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.033182197007389065), 'Bgr': np.float64(0.006839079760899626), 'Bu': np.float64(0.038219461207325625), 'Sc': np.float64(0.005208672083258821), 'Sod': np.float64(0.0), 'Pot': np.float64(0.0), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.00796977935943589), 'Wbcc': np.float64(0.0), 'Rbcc': np.float64(0.0013812745559526353), 'Htn': np.float64(0.0), 'Dm': np.float64(0.002901034234846307), 'Cad': np.fl

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead o

In [17]:
# =============================================================================
# STEP 15: INSPECT DODA RANK CHANGES
# =============================================================================

INSPECTION_TOP_K = 10

selector = rank_doda_results[INSPECTION_TOP_K]["selector"]

print("=" * 70)
print(f"DODA RANK COMPARISON — TOP-{INSPECTION_TOP_K}")
print("=" * 70)

rank_comparison = selector.compare_scores()

display(rank_comparison)

print("\nSelected features:")
print(rank_doda_results[INSPECTION_TOP_K]["features"])


DODA RANK COMPARISON — TOP-10

MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      Bu          2              3           1            1
1      Bp          1              6           2           -1
2      Sc          9              2           3            6
3      Su          3             16           4           -1
4     Ane          5             11           5            0
5      Al         14              1           6            8
6     Pcv          7             10           7            0
7      Sg         11              4           8            3
8      Dm         10              7           9            1
9     Bgr          8              9          10           -2


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,Bu,2,0.2348,3,0.90,1,0.0323,1
1,Bp,1,1.0000,6,0.80,2,0.0320,-1
2,Sc,9,0.0320,2,1.00,3,0.0309,6
3,Su,3,0.2078,16,0.55,4,0.0306,-1
4,Ane,5,0.1002,11,0.70,5,0.0305,0
5,Al,14,0.0000,1,1.00,6,0.0301,8
6,Pcv,7,0.0490,10,0.70,7,0.0301,0
7,Sg,11,0.0154,4,0.85,8,0.0300,3
8,Dm,10,0.0178,7,0.80,9,0.0299,1
9,Bgr,8,0.0420,9,0.70,10,0.0299,-2



Selected features:
['Bu', 'Bp', 'Sc', 'Su', 'Ane', 'Al', 'Pcv', 'Sg', 'Dm', 'Bgr']


In [18]:
# =============================================================================
# STEP 15B: EXPLICIT LASSO vs DODA RANK-SHIFT TABLE
# =============================================================================
# This table separates:
#   - statistical LASSO rank
#   - DODA final rank
#   - clinical weight
#   - rank movement
#
# Positive Rank_Change means the feature moved upward.
# Example: LASSO rank 10 -> DODA rank 4 = +6
#
# IMPORTANT:
# The selector is explicitly retrieved from rank_doda_results
# for the selected Top-K value. This avoids relying on a
# previously defined 'selector' variable.

# -------------------------------------------------------------------------
# Select which DODA Top-K experiment to inspect
# -------------------------------------------------------------------------

k = 10

if k not in rank_doda_results:
    raise KeyError(
        f"Top-K={k} was not found in rank_doda_results. "
        f"Available values: {list(rank_doda_results.keys())}"
    )

selector = rank_doda_results[k]["selector"]


# -------------------------------------------------------------------------
# LASSO / mathematical ranking
# -------------------------------------------------------------------------

math_rank = pd.DataFrame(
    selector.math_scores_.items(),
    columns=[
        "Feature",
        "LASSO_Normalized_Score"
    ]
)

math_rank["LASSO_Rank"] = (
    math_rank["LASSO_Normalized_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# DODA final ranking
# -------------------------------------------------------------------------

final_rank = pd.DataFrame(
    selector.final_scores_.items(),
    columns=[
        "Feature",
        "DODA_Final_Score"
    ]
)

final_rank["DODA_Rank"] = (
    final_rank["DODA_Final_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# Combine rankings
# -------------------------------------------------------------------------

rank_shift_df = (
    math_rank[
        [
            "Feature",
            "LASSO_Normalized_Score",
            "LASSO_Rank"
        ]
    ]
    .merge(
        final_rank[
            [
                "Feature",
                "DODA_Final_Score",
                "DODA_Rank"
            ]
        ],
        on="Feature",
        how="inner"
    )
)


# -------------------------------------------------------------------------
# Add clinical weights
# -------------------------------------------------------------------------

rank_shift_df["Clinical_Weight"] = (
    rank_shift_df["Feature"]
    .map(selector.clinical_weights_)
)


# -------------------------------------------------------------------------
# Calculate rank movement
# -------------------------------------------------------------------------
# Positive value  = moved upward
# Negative value  = moved downward
# Zero             = no rank movement

rank_shift_df["Rank_Change"] = (
    rank_shift_df["LASSO_Rank"]
    - rank_shift_df["DODA_Rank"]
)


# -------------------------------------------------------------------------
# Sort by DODA final rank
# -------------------------------------------------------------------------

rank_shift_df = (
    rank_shift_df
    .sort_values(
        by="DODA_Rank"
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

print("=" * 80)
print(f"LASSO vs DODA RANK-SHIFT ANALYSIS — TOP-{k}")
print("=" * 80)

display(rank_shift_df)

LASSO vs DODA RANK-SHIFT ANALYSIS — TOP-10


,Feature,LASSO_Normalized_Score,LASSO_Rank,DODA_Final_Score,DODA_Rank,Clinical_Weight,Rank_Change
0,Bu,0.234835,2,0.032258,1,0.90,1
1,Bp,1.000000,1,0.032018,2,0.80,-1
2,Sc,0.032004,9,0.030886,3,1.00,6
3,Su,0.207790,3,0.030579,4,0.55,-1
4,Ane,0.100181,5,0.030536,5,0.70,0
5,Al,0.000000,13,0.030092,6,1.00,7
6,Pcv,0.048969,7,0.030077,7,0.70,0
7,Sg,0.015377,11,0.029958,8,0.85,3
8,Dm,0.017825,10,0.029911,9,0.80,1
9,Bgr,0.042022,8,0.029857,10,0.70,-2


In [19]:
# =============================================================================
# STEP 16: INSPECT CLINICAL WEIGHTS
# =============================================================================

clinical_weight_table = pd.DataFrame({
    "Feature": list(selector.clinical_weights_.keys()),
    "Clinical_Weight": list(selector.clinical_weights_.values())
}).sort_values(
    "Clinical_Weight",
    ascending=False
).reset_index(drop=True)

display(clinical_weight_table)


,Feature,Clinical_Weight
0,Al,1.00
1,Sc,1.00
2,Bu,0.90
3,Sg,0.85
4,Htn,0.85
5,Dm,0.80
6,Bp,0.80
7,Hemo,0.75
8,Pcv,0.70
9,Ane,0.70


In [20]:
# =============================================================================
# STEP 17: DODA MODEL EVALUATION — FIXED SPLIT
# =============================================================================

fixed_doda_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rank_doda_results[top_k]["X_train"]
    Xte_selected = rank_doda_results[top_k]["X_test"]
    selected_features = rank_doda_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_doda_rows.append({
            "Method": "DODA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_doda_results_df = pd.DataFrame(fixed_doda_rows)
display(fixed_doda_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,DODA,5,Logistic Regression,"Bu, Bp, Sc, Su, Ane",0.5303,0.5378,0.6275,0.5792,0.5518
1,DODA,5,Random Forest,"Bu, Bp, Sc, Su, Ane",0.4899,0.5053,0.4706,0.4873,0.4981
2,DODA,5,XGBoost,"Bu, Bp, Sc, Su, Ane",0.5101,0.5229,0.5588,0.5403,0.5008
3,DODA,10,Logistic Regression,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr",0.5455,0.5556,0.5882,0.5714,0.5564
4,DODA,10,Random Forest,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr",0.4697,0.4842,0.4510,0.4670,0.5152
5,DODA,10,XGBoost,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr",0.5354,0.5510,0.5294,0.5400,0.5329
6,DODA,15,Logistic Regression,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr, Ba,...",0.5404,0.5495,0.5980,0.5728,0.5716
7,DODA,15,Random Forest,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr, Ba,...",0.5657,0.5800,0.5686,0.5743,0.5659
8,DODA,15,XGBoost,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr, Ba,...",0.5606,0.5789,0.5392,0.5584,0.5558
9,DODA,20,Logistic Regression,"Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr, Ba,...",0.5859,0.5943,0.6176,0.6058,0.5887


In [21]:
# =============================================================================
# STEP 18: FIXED-SPLIT LASSO vs DODA COMPARISON
# =============================================================================

fixed_comparison_df = pd.concat(
    [fixed_lasso_results_df, fixed_doda_results_df],
    ignore_index=True
)

display(fixed_comparison_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,LASSO,5,Logistic Regression,"Bp, Bu, Su, Ba, Ane",0.5303,0.5391,0.6078,0.5714,0.5587
1,LASSO,5,Random Forest,"Bp, Bu, Su, Ba, Ane",0.5354,0.5532,0.5098,0.5306,0.5628
2,LASSO,5,XGBoost,"Bp, Bu, Su, Ba, Ane",0.5455,0.5577,0.5686,0.5631,0.5288
3,LASSO,10,Logistic Regression,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm",0.5455,0.5517,0.6275,0.5872,0.5642
4,LASSO,10,Random Forest,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm",0.5505,0.5644,0.5588,0.5616,0.5575
5,LASSO,10,XGBoost,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm",0.5253,0.5385,0.5490,0.5437,0.5239
6,LASSO,15,Logistic Regression,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.5758,0.5818,0.6275,0.6038,0.5774
7,LASSO,15,Random Forest,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.5101,0.5234,0.5490,0.5359,0.5283
8,LASSO,15,XGBoost,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.4545,0.4722,0.5000,0.4857,0.4861
9,LASSO,20,Logistic Regression,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...",0.5657,0.5755,0.5980,0.5865,0.5751


In [22]:
# =============================================================================
# STEP 19: COMPARE FEATURE SETS ON THE FIXED SPLIT
# =============================================================================

fixed_feature_comparison = []

for top_k in TOP_K_VALUES:

    lasso_set = set(lasso_results[top_k]["features"])
    doda_set = set(rank_doda_results[top_k]["features"])

    intersection = lasso_set & doda_set
    union = lasso_set | doda_set

    jaccard = len(intersection) / len(union)

    fixed_feature_comparison.append({
        "Top_K": top_k,
        "LASSO_Features": ", ".join(lasso_results[top_k]["features"]),
        "DODA_Features": ", ".join(rank_doda_results[top_k]["features"]),
        "Same_Feature_Set": lasso_set == doda_set,
        "Overlap_Count": len(intersection),
        "Jaccard": jaccard,
        "Features_Changed": top_k - len(intersection)
    })

fixed_feature_comparison_df = pd.DataFrame(fixed_feature_comparison)

display(fixed_feature_comparison_df)


,Top_K,LASSO_Features,DODA_Features,Same_Feature_Set,Overlap_Count,Jaccard,Features_Changed
0,5,"Bp, Bu, Su, Ba, Ane","Bu, Bp, Sc, Su, Ane",False,4,0.666667,1
1,10,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm","Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr",False,8,0.666667,2
2,15,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...","Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr, Ba,...",False,11,0.578947,4
3,20,"Bp, Bu, Su, Ba, Ane, Appet, Pcv, Bgr, Sc, Dm, ...","Bu, Bp, Sc, Su, Ane, Al, Pcv, Sg, Dm, Bgr, Ba,...",False,18,0.818182,2


In [23]:
# =============================================================================
# STEP 20: SAVE FIXED-SPLIT DODA RESULTS
# =============================================================================

doda_dir = Path("../../../results/kidney_disease/doda")
doda_dir.mkdir(parents=True, exist_ok=True)

fixed_doda_results_df.to_csv(
    doda_dir / "lasso_doda_fixed_split_results.csv",
    index=False
)

fixed_feature_comparison_df.to_csv(
    doda_dir / "lasso_doda_fixed_split_feature_comparison.csv",
    index=False
)

print(f"Saved results to: {doda_dir.resolve()}")


Saved results to: C:\Users\johnm\msc_research\results\kidney_disease\doda


# 3.1 Five-Fold Stratified Cross-Validation

The fixed 80/20 split is followed by a separate **5-fold stratified cross-validation** experiment.

Feature selection is refit inside each training fold. DODA also receives all features and performs the final Top-K selection inside the training fold. Logistic Regression scaling is fitted only on the corresponding training fold.

The DODA fusion method remains **RankFusion**, matching the current BD-KDD pipeline.

In [24]:
# =============================================================================
# STEP 21: 5-FOLD CV CONFIGURATION
# =============================================================================
from sklearn.model_selection import StratifiedKFold

CV5_SPLITS = 5
CV5_RANDOM_STATE = 42
cv5 = StratifiedKFold(n_splits=CV5_SPLITS, shuffle=True, random_state=CV5_RANDOM_STATE)
cv5_results = []

print("5-fold CV configuration loaded.")
print("Splits:", CV5_SPLITS)
print("Top-K values:", TOP_K_VALUES)
print("Fusion: RankFusion")


5-fold CV configuration loaded.
Splits: 5
Top-K values: [5, 10, 15, 20]
Fusion: RankFusion


In [25]:
# =============================================================================
# STEP 22: 5-FOLD CV — LASSO vs DODA
# =============================================================================

# Reset results before running the CV.
# This prevents duplicate rows if the cell is executed more than once.
cv5_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv5.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    # -------------------------------------------------------------------------
    # Scale ONLY using the training-fold statistics before LASSO selection.
    # -------------------------------------------------------------------------

    fold_scaler = StandardScaler()

    X_train_fold_scaled = pd.DataFrame(
        fold_scaler.fit_transform(X_train_fold),
        columns=X_train_fold.columns,
        index=X_train_fold.index
    )

    X_val_fold_scaled = pd.DataFrame(
        fold_scaler.transform(X_val_fold),
        columns=X_val_fold.columns,
        index=X_val_fold.index
    )

    # -------------------------------------------------------------------------
    # LASSO ranking is learned ONLY from the training fold.
    # -------------------------------------------------------------------------

    lasso_fold = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.1,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    lasso_fold.fit(
        X_train_fold_scaled,
        y_train_fold
    )

    lasso_fold_scores = pd.DataFrame({
        "Feature": X_train_fold.columns,
        "Score": np.abs(lasso_fold.coef_[0])
    }).sort_values(
        "Score",
        ascending=False
    )

    # -------------------------------------------------------------------------
    # Evaluate each Top-K
    # -------------------------------------------------------------------------

    for top_k in TOP_K_VALUES:

        # -------------------------
        # Baseline LASSO Top-K
        # -------------------------

        lasso_features = (
            lasso_fold_scores
            .head(top_k)["Feature"]
            .tolist()
        )

        # -------------------------
        # DODA
        # -------------------------

        fold_doda = DODASelector(
            operators=[
                SklearnAdapter(
                    LogisticRegression(
                        penalty="l1",
                        solver="liblinear",
                        C=0.1,
                        max_iter=2000,
                        class_weight="balanced",
                        random_state=42
                    )
                )
            ],
            provider=JSONProvider(
                str(WEIGHTS_PATH)
            ),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold_scaled,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        # -------------------------
        # Compare both methods
        # -------------------------

        feature_sets = {
            "LASSO": lasso_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            # -------------------------------------------------------------
            # Downstream models
            # -------------------------------------------------------------

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                # Logistic Regression is scaled using training-fold
                # statistics only.
                if model_name == "Logistic Regression":

                    scaler = StandardScaler()

                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                # ---------------------------------------------------------
                # Train
                # ---------------------------------------------------------

                model.fit(
                    Xtr,
                    y_train_fold
                )

                # ---------------------------------------------------------
                # Predict
                # ---------------------------------------------------------

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                # ---------------------------------------------------------
                # Store results
                # ---------------------------------------------------------

                cv5_results.append({
                    "Fold": fold,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,

                    "Accuracy": accuracy_score(
                        y_val_fold,
                        y_pred
                    ),

                    "Precision": precision_score(
                        y_val_fold,
                        y_pred,
                        zero_division=0
                    ),

                    "Recall": recall_score(
                        y_val_fold,
                        y_pred,
                        zero_division=0
                    ),

                    "F1": f1_score(
                        y_val_fold,
                        y_pred,
                        zero_division=0
                    ),

                    "ROC_AUC": roc_auc_score(
                        y_val_fold,
                        y_prob
                    ),

                    "Selected_Features": ", ".join(
                        selected_features
                    )
                })


# =============================================================================
# VALIDATION
# =============================================================================

cv5_results_df = pd.DataFrame(cv5_results)

expected_rows = (
    5
    * len(TOP_K_VALUES)
    * 2
    * len(models)
)

print("Expected rows:", expected_rows)
print("Actual rows:  ", len(cv5_results_df))

assert len(cv5_results_df) == expected_rows

print(
    "\nCompleted 5-fold CV. Rows:",
    len(cv5_results_df)
)

display(
    cv5_results_df.head()
)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7F97DBB10>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.003865822342735337), 'Bp': np.float64(0.15778162552295807), 'Sg': np.float64(0.0831185017659493), 'Al': np.float64(0.0), 'Su': np.float64(0.042429847718452714), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.026436198756198334), 'Bgr': np.float64(0.015040512081841886), 'Bu': np.float64(0.0), 'Sc': np.float64(0.011541437819182819), 'Sod': np.float64(0.0032811826464089113), 'Pot': np.float64(0.019272522339605993), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.0), 'Wbcc': np.float64(0.016593647253582967), 'Rbcc': np.float64(0.03651693953265173), 'Htn': np.float64(0.0), 'Dm': np.float64(0.0

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead o

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7F97DBD90>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.003865822342735337), 'Bp': np.float64(0.15778162552295807), 'Sg': np.float64(0.0831185017659493), 'Al': np.float64(0.0), 'Su': np.float64(0.042429847718452714), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.026436198756198334), 'Bgr': np.float64(0.015040512081841886), 'Bu': np.float64(0.0), 'Sc': np.float64(0.011541437819182819), 'Sod': np.float64(0.0032811826464089113), 'Pot': np.float64(0.019272522339605993), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.0), 'Wbcc': np.float64(0.016593647253582967), 'Rbcc': np.float64(0.03651693953265173), 'Htn': np.float64(0.0), 'Dm': np.float64(0.0

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7F981C640>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.003865822342735337), 'Bp': np.float64(0.15778162552295807), 'Sg': np.float64(0.0831185017659493), 'Al': np.float64(0.0), 'Su': np.float64(0.042429847718452714), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.026436198756198334), 'Bgr': np.float64(0.015040512081841886), 'Bu': np.float64(0.0), 'Sc': np.float64(0.011541437819182819), 'Sod': np.float64(0.0032811826464089113), 'Pot': np.float64(0.019272522339605993), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.0), 'Wbcc': np.float64(0.016593647253582967), 'Rbcc': np.float64(0.03651693953265173), 'Htn': np.float64(0.0), 'Dm': np.float64(0.0

...

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7FE8C5DB0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.0), 'Bp': np.float64(0.1384746815978939), 'Sg': np.float64(0.009514100212471078), 'Al': np.float64(0.0), 'Su': np.float64(0.09234427580585562), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.004109687838853666), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.037752483215554224), 'Bgr': np.float64(0.0), 'Bu': np.float64(0.06628813906633899), 'Sc': np.float64(0.0), 'Sod': np.float64(0.0), 'Pot': np.float64(0.0021584693694957514), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.09095231740578455), 'Wbcc': np.float64(0.0), 'Rbcc': np.float64(0.009582138739960682), 'Htn': np.float64(0.0026932345717025716), 'Dm': np.float64(0.0), 'Cad': np.floa

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Expected rows: 120
Actual rows:   120

Completed 5-fold CV. Rows: 120


,Fold,Method,Top_K,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Selected_Features
0,1,LASSO,5,Logistic Regression,0.525253,0.530973,0.594059,0.560748,0.524497,"Bp, Sg, Su, Rbcc, Ba"
1,1,LASSO,5,Random Forest,0.525253,0.533333,0.554455,0.543689,0.513831,"Bp, Sg, Su, Rbcc, Ba"
2,1,LASSO,5,XGBoost,0.520202,0.526786,0.584158,0.553991,0.499898,"Bp, Sg, Su, Rbcc, Ba"
3,1,DODA,5,Logistic Regression,0.500000,0.509615,0.524752,0.517073,0.518322,"Bp, Sg, Sc, Su, Rbcc"
4,1,DODA,5,Random Forest,0.540404,0.550000,0.544554,0.547264,0.548382,"Bp, Sg, Sc, Su, Rbcc"


In [26]:
# =============================================================================
# STEP 23: 5-FOLD CV PERFORMANCE SUMMARY
# =============================================================================
cv5_summary_df = (
    cv5_results_df.groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)
cv5_summary_df.columns = [
    "Method", "Top_K", "Model", "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD", "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD", "ROC_AUC_Mean", "ROC_AUC_STD"
]
display(cv5_summary_df.round(4))

cv5_dir = Path("../../../results/kidney_disease/cv")
cv5_dir.mkdir(parents=True, exist_ok=True)
cv5_results_df.to_csv(cv5_dir / "lasso_doda_5fold_results.csv", index=False)
cv5_summary_df.to_csv(cv5_dir / "lasso_doda_5fold_summary.csv", index=False)


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,DODA,5,Logistic Regression,0.5102,0.0302,0.5198,0.0243,0.5761,0.0597,0.5460,0.0391,0.5129,0.0231
1,DODA,5,Random Forest,0.5101,0.0389,0.5236,0.0390,0.5424,0.0164,0.5323,0.0247,0.5019,0.0413
2,DODA,5,XGBoost,0.5040,0.0188,0.5166,0.0181,0.5286,0.0287,0.5222,0.0191,0.4992,0.0325
3,DODA,10,Logistic Regression,0.5283,0.0149,0.5366,0.0110,0.5898,0.0387,0.5617,0.0217,0.5258,0.0179
4,DODA,10,Random Forest,0.4989,0.0518,0.5138,0.0498,0.5307,0.0455,0.5210,0.0400,0.4969,0.0471
5,DODA,10,XGBoost,0.5111,0.0237,0.5235,0.0216,0.5385,0.0433,0.5301,0.0261,0.5070,0.0494
6,DODA,15,Logistic Regression,0.5243,0.0077,0.5337,0.0073,0.5799,0.0153,0.5557,0.0075,0.5296,0.0187
7,DODA,15,Random Forest,0.5081,0.0249,0.5204,0.0233,0.5366,0.0483,0.5275,0.0297,0.5197,0.0391
8,DODA,15,XGBoost,0.5162,0.0403,0.5281,0.0369,0.5464,0.0351,0.5370,0.0349,0.5271,0.0531
9,DODA,20,Logistic Regression,0.5233,0.0112,0.5326,0.0100,0.5799,0.0204,0.5552,0.0125,0.5308,0.0165


# 3.2 Robustness: 5×5 Repeated Stratified Cross-Validation

The fixed split provides a reproducible initial comparison.

For robustness, feature selection is **recomputed inside every training fold**. This prevents information from the validation fold from influencing LASSO or DODA.

There are:

- 5 folds
- 5 repeats
- 25 train/validation runs
- 4 Top-K budgets
- 2 feature-selection methods
- 3 downstream models

This produces 600 model-evaluation rows.


In [27]:
# =============================================================================
# STEP 24: REPEATED CV CONFIGURATION
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold

TOP_K_VALUES = [5, 10, 15, 20]

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

print(f"Top-K values: {TOP_K_VALUES}")
print(f"CV runs: {N_SPLITS * N_REPEATS}")


Top-K values: [5, 10, 15, 20]
CV runs: 25


In [28]:
# =============================================================================
# STEP 25: 5×5 REPEATED CV — LASSO vs DODA
# =============================================================================

cv_results = []

for run_id, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    print(f"Run {run_id:02d}/{N_SPLITS * N_REPEATS}")

    # -----------------------------------------------------------------
    # Scale ONLY the training partition before LASSO selection.
    # -----------------------------------------------------------------

    fold_scaler = StandardScaler()

    X_train_fold_scaled = pd.DataFrame(
        fold_scaler.fit_transform(X_train_fold),
        columns=X_train_fold.columns,
        index=X_train_fold.index
    )

    X_val_fold_scaled = pd.DataFrame(
        fold_scaler.transform(X_val_fold),
        columns=X_val_fold.columns,
        index=X_val_fold.index
    )

    # -----------------------------------------------------------------
    # LASSO scores ALL features inside this training partition.
    # -----------------------------------------------------------------

    lasso_all = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=0.1,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )

    lasso_all.fit(
        X_train_fold_scaled,
        y_train_fold
    )

    lasso_fold_scores = pd.DataFrame({
        "Feature": X_train_fold.columns,
        "Score": np.abs(lasso_all.coef_[0])
    }).sort_values(
        "Score",
        ascending=False
    )

    # -----------------------------------------------------------------
    # DODA also receives ALL features and performs final Top-K.
    # -----------------------------------------------------------------

    for top_k in TOP_K_VALUES:

        # -------------------------
        # LASSO Top-K
        # -------------------------

        lasso_features = (
            lasso_fold_scores
            .head(top_k)["Feature"]
            .tolist()
        )

        # -------------------------
        # DODA Top-K
        # -------------------------

        fold_doda_operator = SklearnAdapter(
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=0.1,
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )

        fold_doda = DODASelector(
            operators=[fold_doda_operator],
            provider=JSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold_scaled,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        # -------------------------
        # Model loop
        # -------------------------

        feature_sets = {
            "LASSO": lasso_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv_results.append({
                    "Run": run_id,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Accuracy": accuracy_score(
                        y_val_fold, y_pred
                    ),
                    "Precision": precision_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "Recall": recall_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "F1": f1_score(
                        y_val_fold, y_pred, zero_division=0
                    ),
                    "ROC_AUC": roc_auc_score(
                        y_val_fold, y_prob
                    ),
                    "Selected_Features": ", ".join(
                        selected_features
                    )
                })

cv_results_df = pd.DataFrame(cv_results)

print("Completed.")
print(f"Rows generated: {len(cv_results_df)}")

assert len(cv_results_df) == 25 * 4 * 2 * 3

display(cv_results_df.head())


Run 01/25
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7F9836690>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.003865822342735337), 'Bp': np.float64(0.15778162552295807), 'Sg': np.float64(0.0831185017659493), 'Al': np.float64(0.0), 'Su': np.float64(0.042429847718452714), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.026436198756198334), 'Bgr': np.float64(0.015040512081841886), 'Bu': np.float64(0.0), 'Sc': np.float64(0.011541437819182819), 'Sod': np.float64(0.0032811826464089113), 'Pot': np.float64(0.019272522339605993), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.0), 'Wbcc': np.float64(0.016593647253582967), 'Rbcc': np.float64(0.03651693953265173), 'Htn': np.float64(0.0), 'Dm': np.f

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead o

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7FE9205D0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.003865822342735337), 'Bp': np.float64(0.15778162552295807), 'Sg': np.float64(0.0831185017659493), 'Al': np.float64(0.0), 'Su': np.float64(0.042429847718452714), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.026436198756198334), 'Bgr': np.float64(0.015040512081841886), 'Bu': np.float64(0.0), 'Sc': np.float64(0.011541437819182819), 'Sod': np.float64(0.0032811826464089113), 'Pot': np.float64(0.019272522339605993), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.0), 'Wbcc': np.float64(0.016593647253582967), 'Rbcc': np.float64(0.03651693953265173), 'Htn': np.float64(0.0), 'Dm': np.float64(0.0

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7FE99D2D0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.003865822342735337), 'Bp': np.float64(0.15778162552295807), 'Sg': np.float64(0.0831185017659493), 'Al': np.float64(0.0), 'Su': np.float64(0.042429847718452714), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.026436198756198334), 'Bgr': np.float64(0.015040512081841886), 'Bu': np.float64(0.0), 'Sc': np.float64(0.011541437819182819), 'Sod': np.float64(0.0032811826464089113), 'Pot': np.float64(0.019272522339605993), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.0), 'Wbcc': np.float64(0.016593647253582967), 'Rbcc': np.float64(0.03651693953265173), 'Htn': np.float64(0.0), 'Dm': np.float64(0.0

...

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000001F7FED221B0>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['Age', 'Bp', 'Sg', 'Al', 'Su', 'Rbc', 'Pc', 'Pcc', 'Ba', 'Bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'Age': np.float64(0.0), 'Bp': np.float64(0.16267731301501193), 'Sg': np.float64(0.08365487301144182), 'Al': np.float64(0.013648784391673427), 'Su': np.float64(0.07674379876456915), 'Rbc': np.float64(0.0), 'Pc': np.float64(0.0), 'Pcc': np.float64(0.0), 'Ba': np.float64(0.022113698451859978), 'Bgr': np.float64(0.006383430063673035), 'Bu': np.float64(0.08862037407644868), 'Sc': np.float64(0.01410442795313359), 'Sod': np.float64(0.0), 'Pot': np.float64(0.004915759017971066), 'Hemo': np.float64(0.0), 'Pcv': np.float64(0.023717016093182372), 'Wbcc': np.float64(0.0), 'Rbcc': np.float64(0.046113271431917915), 'Htn': np.float64(0.0), 'Dm': np.float64(0.0),

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Completed.
Rows generated: 600


,Run,Method,Top_K,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Selected_Features
0,1,LASSO,5,Logistic Regression,0.525253,0.530973,0.594059,0.560748,0.524497,"Bp, Sg, Su, Rbcc, Ba"
1,1,LASSO,5,Random Forest,0.525253,0.533333,0.554455,0.543689,0.513831,"Bp, Sg, Su, Rbcc, Ba"
2,1,LASSO,5,XGBoost,0.520202,0.526786,0.584158,0.553991,0.499898,"Bp, Sg, Su, Rbcc, Ba"
3,1,DODA,5,Logistic Regression,0.500000,0.509615,0.524752,0.517073,0.518322,"Bp, Sg, Sc, Su, Rbcc"
4,1,DODA,5,Random Forest,0.540404,0.550000,0.544554,0.547264,0.548382,"Bp, Sg, Sc, Su, Rbcc"


In [29]:
# =============================================================================
# STEP 26: CV PERFORMANCE SUMMARY
# =============================================================================

cv_summary_df = (
    cv_results_df
    .groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)

cv_summary_df.columns = [
    "Method", "Top_K", "Model",
    "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD",
    "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD",
    "ROC_AUC_Mean", "ROC_AUC_STD"
]

display(cv_summary_df.round(4))


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,DODA,5,Logistic Regression,0.5095,0.0336,0.5193,0.0285,0.5724,0.0611,0.5439,0.0411,0.5142,0.0338
1,DODA,5,Random Forest,0.5107,0.0345,0.5229,0.0328,0.5384,0.0473,0.5299,0.0355,0.5087,0.0445
2,DODA,5,XGBoost,0.5067,0.0347,0.5176,0.0322,0.5348,0.0664,0.5252,0.0467,0.5060,0.0408
3,DODA,10,Logistic Regression,0.5170,0.0252,0.5273,0.0229,0.5712,0.0410,0.5478,0.0271,0.5173,0.0308
4,DODA,10,Random Forest,0.5109,0.0419,0.5234,0.0396,0.5416,0.0536,0.5315,0.0413,0.5100,0.0396
5,DODA,10,XGBoost,0.5079,0.0364,0.5198,0.0343,0.5381,0.0513,0.5282,0.0399,0.5106,0.0462
6,DODA,15,Logistic Regression,0.5178,0.0265,0.5283,0.0251,0.5673,0.0385,0.5467,0.0275,0.5209,0.0353
7,DODA,15,Random Forest,0.5115,0.0296,0.5234,0.0276,0.5389,0.0443,0.5305,0.0327,0.5187,0.0373
8,DODA,15,XGBoost,0.5180,0.0324,0.5294,0.0303,0.5487,0.0489,0.5383,0.0352,0.5243,0.0403
9,DODA,20,Logistic Regression,0.5168,0.0305,0.5276,0.0291,0.5626,0.0454,0.5439,0.0326,0.5236,0.0327


In [30]:
# =============================================================================
# STEP 24: SAVE REPEATED-CV RESULTS
# =============================================================================

cv_dir = Path("../../../results/kidney_disease/cv")
cv_dir.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(
    cv_dir / "lasso_vs_doda_5x5_repeated_cv_results.csv",
    index=False
)

cv_summary_df.to_csv(
    cv_dir / "lasso_vs_doda_5x5_repeated_cv_summary.csv",
    index=False
)

print(f"Saved to: {cv_dir.resolve()}")


Saved to: C:\Users\johnm\msc_research\results\kidney_disease\cv


# 4. Feature-Selection Stability, Rank/Set Change, and Statistical Comparison

Two different concepts are reported:

- **Feature-set change:** whether the Top-K membership changes.
- **Rank change:** whether feature ordering changes even when membership is retained.

Jaccard similarity evaluates set overlap, so it does not measure ordering. The fixed-split DODA rank comparison is used to inspect ordering changes.


In [31]:
# =============================================================================
# STEP 28: FEATURE-SET COMPARISON ACROSS THE 25 CV RUNS
# =============================================================================

feature_comparison_rows = []

for run_id in range(1, 26):

    for top_k in TOP_K_VALUES:

        lasso_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "LASSO") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        doda_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        lasso_set = set(
            lasso_row["Selected_Features"].split(", ")
        )
        doda_set = set(
            doda_row["Selected_Features"].split(", ")
        )

        overlap = len(lasso_set & doda_set)
        union = len(lasso_set | doda_set)

        feature_comparison_rows.append({
            "Run": run_id,
            "Top_K": top_k,
            "Same_Feature_Set": lasso_set == doda_set,
            "Overlap_Count": overlap,
            "Jaccard": overlap / union,
            "Features_Changed": top_k - overlap
        })

feature_comparison_df = pd.DataFrame(
    feature_comparison_rows
)

display(
    feature_comparison_df
    .groupby("Top_K")
    [["Same_Feature_Set", "Overlap_Count", "Jaccard", "Features_Changed"]]
    .agg(["mean", "std"])
    .round(4)
)


Same_Feature_Set      Overlap_Count         Jaccard          \
                  mean  std          mean     std    mean     std   
Top_K                                                               
5                  0.0  0.0          3.44  0.5831  0.5357  0.1331   
10                 0.0  0.0          7.24  1.0909  0.5783  0.1339   
15                 0.0  0.0         10.64  1.0360  0.5539  0.0850   
20                 0.0  0.0         17.52  0.5099  0.7802  0.0403   

      Features_Changed          
                  mean     std  
Top_K                           
5                 1.56  0.5831  
10                2.76  1.0909  
15                4.36  1.0360  
20                2.48  0.5099

In [32]:
# =============================================================================
# STEP 29: FEATURE-SELECTION STABILITY
# =============================================================================

from itertools import combinations

stability_rows = []

for method in ["LASSO", "DODA"]:

    for top_k in TOP_K_VALUES:

        subset = cv_results_df[
            (cv_results_df["Method"] == method) &
            (cv_results_df["Top_K"] == top_k)
        ]

        # One feature set per run; model does not affect selection.
        run_sets = {}

        for run_id in range(1, 26):
            row = subset[
                subset["Run"] == run_id
            ].iloc[0]

            run_sets[run_id] = set(
                row["Selected_Features"].split(", ")
            )

        pairwise_jaccards = []

        for a, b in combinations(range(1, 26), 2):

            set_a = run_sets[a]
            set_b = run_sets[b]

            pairwise_jaccards.append(
                len(set_a & set_b) / len(set_a | set_b)
            )

        stability_rows.append({
            "Method": method,
            "Top_K": top_k,
            "Mean_Jaccard": np.mean(pairwise_jaccards),
            "SD_Jaccard": np.std(pairwise_jaccards, ddof=1),
            "Min_Jaccard": np.min(pairwise_jaccards),
            "Max_Jaccard": np.max(pairwise_jaccards)
        })

stability_df = pd.DataFrame(stability_rows)

display(stability_df.round(4))


,Method,Top_K,Mean_Jaccard,SD_Jaccard,Min_Jaccard,Max_Jaccard
0,LASSO,5,0.3951,0.1662,0.1111,1.0000
1,LASSO,10,0.4770,0.1266,0.1765,0.8182
2,LASSO,15,0.6222,0.1002,0.4286,1.0000
3,LASSO,20,0.8729,0.0614,0.7391,1.0000
4,DODA,5,0.3884,0.1645,0.1111,1.0000
5,DODA,10,0.6029,0.1050,0.4286,0.8182
6,DODA,15,0.7527,0.0852,0.5789,1.0000
7,DODA,20,0.9096,0.0593,0.7391,1.0000


## 4.1 Additional Feature-Selection Stability Analyses

The 25 repeated-CV runs are used to quantify selection frequency, chance-adjusted stability, and direct LASSO–DODA feature-set agreement.

In [33]:
# =============================================================================
# STEP 30: FEATURE SELECTION FREQUENCY
# =============================================================================
frequency_rows = []
for method in ["LASSO", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        counts = {feature: 0 for feature in X.columns}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            for feature in row["Selected_Features"].split(", "):
                counts[feature] += 1
        for feature, count in counts.items():
            frequency_rows.append({
                "Method": method, "Top_K": top_k, "Feature": feature,
                "Selection_Count": count,
                "Selection_Frequency": count / (N_SPLITS * N_REPEATS)
            })
feature_frequency_df = pd.DataFrame(frequency_rows)

display(
    feature_frequency_df.sort_values(
        ["Method", "Top_K", "Selection_Frequency", "Feature"],
        ascending=[True, True, False, True]
    ).groupby(["Method", "Top_K"], group_keys=False).head(10).round(4)
)


,Method,Top_K,Feature,Selection_Count,Selection_Frequency
97,DODA,5,Bp,25,1.00
106,DODA,5,Bu,17,0.68
111,DODA,5,Pcv,15,0.60
107,DODA,5,Sc,14,0.56
99,DODA,5,Al,12,0.48
...,...,...,...,...,...
82,LASSO,20,Bu,25,1.00
86,LASSO,20,Hemo,25,1.00
78,LASSO,20,Pc,25,1.00
79,LASSO,20,Pcc,25,1.00


In [34]:
# =============================================================================
# STEP 31: KUNCHEVA INDEX
# =============================================================================
from itertools import combinations
N_FEATURES = X.shape[1]
kuncheva_rows = []

for method in ["LASSO", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        run_sets = {}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            run_sets[run_id] = set(row["Selected_Features"].split(", "))

        for run_a, run_b in combinations(sorted(run_sets), 2):
            intersection = len(run_sets[run_a] & run_sets[run_b])
            denominator = top_k * (N_FEATURES - top_k)
            value = np.nan if denominator == 0 else ((intersection * N_FEATURES) - (top_k ** 2)) / denominator
            kuncheva_rows.append({
                "Method": method, "Top_K": top_k, "Run_A": run_a, "Run_B": run_b,
                "Intersection": intersection, "Kuncheva_Index": value
            })

kuncheva_df = pd.DataFrame(kuncheva_rows)
kuncheva_summary_df = (
    kuncheva_df.groupby(["Method", "Top_K"])["Kuncheva_Index"]
    .agg(Mean_Kuncheva="mean", Std_Kuncheva="std", Min_Kuncheva="min", Max_Kuncheva="max")
    .reset_index()
)
display(kuncheva_summary_df.round(4))


,Method,Top_K,Mean_Kuncheva,Std_Kuncheva,Min_Kuncheva,Max_Kuncheva
0,DODA,5,0.4189,0.2112,-0.0105,1.0000
1,DODA,10,0.5663,0.1391,0.3143,0.8286
2,DODA,15,0.6166,0.1481,0.2889,1.0000
3,DODA,20,0.7100,0.1960,0.1000,1.0000
4,LASSO,5,0.4282,0.2046,-0.0105,1.0000
5,LASSO,10,0.3760,0.2000,-0.2000,0.8286
6,LASSO,15,0.3665,0.2025,-0.0667,1.0000
7,LASSO,20,0.5860,0.2101,0.1000,1.0000


In [35]:
# =============================================================================
# STEP 32: LASSO–DODA FEATURE-SET AGREEMENT SUMMARY
# =============================================================================
agreement_summary_df = (
    feature_comparison_df.groupby("Top_K")
    .agg(
        Same_Feature_Set_Rate=("Same_Feature_Set", "mean"),
        Mean_Overlap=("Overlap_Count", "mean"),
        SD_Overlap=("Overlap_Count", "std"),
        Mean_Jaccard=("Jaccard", "mean"),
        SD_Jaccard=("Jaccard", "std"),
        Mean_Features_Changed=("Features_Changed", "mean"),
        SD_Features_Changed=("Features_Changed", "std")
    ).reset_index()
)
display(agreement_summary_df.round(4))


,Top_K,Same_Feature_Set_Rate,Mean_Overlap,SD_Overlap,Mean_Jaccard,SD_Jaccard,Mean_Features_Changed,SD_Features_Changed
0,5,0.0,3.44,0.5831,0.5357,0.1331,1.56,0.5831
1,10,0.0,7.24,1.0909,0.5783,0.1339,2.76,1.0909
2,15,0.0,10.64,1.0360,0.5539,0.0850,4.36,1.0360
3,20,0.0,17.52,0.5099,0.7802,0.0403,2.48,0.5099


In [36]:
# =============================================================================
# STEP 33: PAIRED STATISTICAL COMPARISON OF PREDICTIVE PERFORMANCE
# =============================================================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

comparison_rows = []

for top_k in TOP_K_VALUES:

    for model_name in models.keys():

        lasso = cv_results_df[
            (cv_results_df["Method"] == "LASSO") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        doda = cv_results_df[
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        assert len(lasso) == 25
        assert len(doda) == 25
        assert (lasso["Run"].values == doda["Run"].values).all()

        for metric in metrics:

            lasso_values = lasso[metric].to_numpy()
            doda_values = doda[metric].to_numpy()

            differences = doda_values - lasso_values

            mean_difference = np.mean(differences)
            sd_difference = np.std(differences, ddof=1)

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    doda_values,
                    lasso_values,
                    alternative="two-sided"
                )

            paired_d = (
                mean_difference / sd_difference
                if sd_difference > 0 else 0.0
            )

            comparison_rows.append({
                "Top_K": top_k,
                "Model": model_name,
                "Metric": metric,
                "LASSO_Mean": np.mean(lasso_values),
                "DODA_Mean": np.mean(doda_values),
                "Mean_Difference_DODA_minus_LASSO": mean_difference,
                "Paired_SD": sd_difference,
                "Paired_Cohens_d": paired_d,
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value
            })

performance_comparison_df = pd.DataFrame(comparison_rows)

reject, adjusted_p, _, _ = multipletests(
    performance_comparison_df["P_Value"],
    alpha=0.05,
    method="holm"
)

performance_comparison_df["Holm_Adjusted_P"] = adjusted_p
performance_comparison_df["Significant"] = reject

display(performance_comparison_df.round(4))


,Top_K,Model,Metric,LASSO_Mean,DODA_Mean,Mean_Difference_DODA_minus_LASSO,Paired_SD,Paired_Cohens_d,Wilcoxon_Statistic,P_Value,Holm_Adjusted_P,Significant
0,5,Logistic Regression,Accuracy,0.5125,0.5095,-0.0030,0.0252,-0.1198,119.0,0.5632,1.0,False
1,5,Logistic Regression,Precision,0.5225,0.5193,-0.0031,0.0221,-0.1428,129.0,0.5485,1.0,False
2,5,Logistic Regression,Recall,0.5720,0.5724,0.0004,0.0508,0.0075,131.0,0.5870,1.0,False
3,5,Logistic Regression,F1,0.5456,0.5439,-0.0017,0.0332,-0.0512,130.0,0.5677,1.0,False
4,5,Logistic Regression,ROC_AUC,0.5163,0.5142,-0.0021,0.0135,-0.1577,142.0,0.5965,1.0,False
5,5,Random Forest,Accuracy,0.5081,0.5107,0.0026,0.0375,0.0693,157.5,0.8930,1.0,False
6,5,Random Forest,Precision,0.5207,0.5229,0.0022,0.0368,0.0611,160.0,0.9578,1.0,False
7,5,Random Forest,Recall,0.5290,0.5384,0.0093,0.0511,0.1830,118.0,0.3604,1.0,False
8,5,Random Forest,F1,0.5240,0.5299,0.0059,0.0352,0.1664,144.0,0.6338,1.0,False
9,5,Random Forest,ROC_AUC,0.5022,0.5087,0.0065,0.0385,0.1676,135.0,0.4742,1.0,False


In [37]:
# =============================================================================
# STEP 34: SAVE STATISTICAL RESULTS
# =============================================================================

stats_dir = Path("../../../results/kidney_disease/statistics")
stats_dir.mkdir(parents=True, exist_ok=True)

feature_comparison_df.to_csv(
    stats_dir / "feature_set_comparison_5x5.csv",
    index=False
)

stability_df.to_csv(
    stats_dir / "feature_selection_stability_5x5.csv",
    index=False
)

performance_comparison_df.to_csv(
    stats_dir / "lasso_vs_doda_predictive_statistics_5x5.csv",
    index=False
)

print(f"Saved statistical outputs to: {stats_dir.resolve()}")

feature_frequency_df.to_csv(
    stats_dir / "feature_selection_frequency_5x5.csv",
    index=False
)

kuncheva_df.to_csv(
    stats_dir / "kuncheva_pairwise_5x5.csv",
    index=False
)

kuncheva_summary_df.to_csv(
    stats_dir / "kuncheva_summary_5x5.csv",
    index=False
)

agreement_summary_df.to_csv(
    stats_dir / "lasso_doda_agreement_summary_5x5.csv",
    index=False
)


Saved statistical outputs to: C:\Users\johnm\msc_research\results\kidney_disease\statistics


In [38]:
# =============================================================================
# STEP 36: FINAL SANITY CHECKS
# =============================================================================

print("=" * 70)
print("FINAL EXPERIMENT SANITY CHECK")
print("=" * 70)

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"5-fold CV result rows: {len(cv5_results_df)}")
print(f"Repeated CV result rows: {len(cv_results_df)}")
print(f"Expected 5-fold rows: {5 * len(TOP_K_VALUES) * 2 * len(models)}")
print(f"Expected repeated CV rows: {25 * len(TOP_K_VALUES) * 2 * len(models)}")

assert X.shape[1] == 24
assert X.isna().sum().sum() == 0
assert TOP_K_VALUES == [5, 10, 15, 20]
assert len(cv5_results_df) == 5 * len(TOP_K_VALUES) * 2 * len(models)
assert len(cv_results_df) == 25 * len(TOP_K_VALUES) * 2 * len(models)
assert len(feature_frequency_df) == 2 * len(TOP_K_VALUES) * X.shape[1]
assert len(kuncheva_summary_df) == 2 * len(TOP_K_VALUES)
assert len(agreement_summary_df) == len(TOP_K_VALUES)

for top_k in TOP_K_VALUES:
    assert len(lasso_results[top_k]["features"]) == top_k
    assert len(rank_doda_results[top_k]["features"]) == top_k

print("All sanity checks passed.")


FINAL EXPERIMENT SANITY CHECK
Total predictors: 24
Top-K values: [5, 10, 15, 20]
Missing values: 0
5-fold CV result rows: 120
Repeated CV result rows: 600
Expected 5-fold rows: 120
Expected repeated CV rows: 600
All sanity checks passed.


# Methodological design notes for the BD-KDD experiment

### 1. Correct predictor count
BD-KDD has **25 processed columns**, but one is the binary target `Class`. Therefore the feature matrix contains **24 predictors**.

### 2. Top-K values
The experiment uses:

**Top-5, Top-10, Top-15, Top-20**

This provides approximately 20%, 40%, 60%, and 80% feature budgets over the 24 predictors. Top-24 is deliberately not used as a selection condition because retaining every feature cannot demonstrate feature-selection behavior.

### 3. Removed imputation
BD-KDD has zero missing values, so no imputation is performed.

### 4. Correct DODA architecture
LASSO is fitted on all 24 standardized predictors inside DODA. DODA receives all 24 statistical scores and performs the final Top-K selection.

### 5. No data leakage in CV
LASSO and DODA are refit inside every training fold. Validation-fold data is never used to calculate feature-selection scores.

### 6. Model-specific scaling
Scaling is applied only to Logistic Regression and is fitted on the training data/fold. Random Forest and XGBoost use the selected features without scaling.

### 7. Consistent result paths
All outputs are stored under:

```text
results/
└── kidney_disease/
    ├── baseline/
    ├── doda/
    ├── cv/
    └── statistics/
```

### 8. Fixed split and repeated CV are separated
The fixed 80/20 experiment is an initial reproducible comparison. The 5×5 repeated stratified CV is the robustness analysis.

### 9. Feature-set change and rank change are separated
Jaccard measures feature-set overlap. DODA's rank-comparison output is retained separately because a feature can change rank without entering or leaving Top-K.

### 10. Clinical weights remain external
The clinical weights are loaded from the pre-specified JSON and are not derived from LASSO scores, target correlations, model performance, or test-fold information.
